# Parte 3 — Ecuaciones elípticas
## 3.7 Existencia para el problema de Dirichlet
### 3.7.01 El método de Perron

Este notebook cubre el apartado **3.7** del temario oficial:

> Existencia de la solución al problema de Dirichlet: el método de Perron.

## Relación con las notas manuscritas

Las notas disponibles no contienen un desarrollo independiente del método de
Perron. Sin embargo, la página 7 fija la convención utilizada en todo este
notebook:

$$
\Delta v\geq0
\quad\Longleftrightarrow\quad
v\text{ es subarmónica}.
$$

También se usan resultados ya transcritos y demostrados en notebooks anteriores:

- propiedad del máximo para funciones subarmónicas;
- principio de comparación;
- desigualdad de Harnack;
- regularidad a partir de la propiedad del promedio;
- barreras y puntos regulares.

Todo lo que no aparece expresamente en las notas se marca como
**Complemento para cerrar el temario oficial**.

## Contenido

1. subfunciones y superfunciones;
2. familias de Perron;
3. máximos y elevación armónica;
4. armonicidad de la envolvente;
5. barreras y recuperación del dato;
6. existencia clásica cuando todos los puntos de frontera son regulares;
7. interpretación conceptual para el Examen General.

## Estado de las fuentes

Las capturas antes enlazadas de las notas, del temario y del Examen General 2023-2 no están incluidas en el repositorio. El desarrollo que sigue es autocontenido. Para cotejar el enunciado oficial debe consultarse el PDF original fuera de este repositorio; no se sustituye aquí por una imagen inventada.


# Simulaciones y visualizaciones

Las celdas se ejecutan directamente. No existe una bandera `VIDEO=True`.

Se incluyen:

1. una familia de subfunciones armónicas en el disco;
2. la envolvente creciente de Perron;
3. comparación con la solución exacta de Poisson;
4. concentración del error cerca de la frontera;
5. una barrera producida por una esfera exterior;
6. recuperación del dato en un punto regular.

CuPy/CUDA se utiliza automáticamente en las mallas densas cuando está disponible.

In [ ]:
from __future__ import annotations

import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import Video, Image, display

BASE = Path("..")
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        GPU_AVAILABLE = True
        print("Backend numérico: CuPy/CUDA")
    else:
        raise RuntimeError("No se encontró un dispositivo CUDA.")
except Exception as exc:
    xp = np
    print("Backend numérico: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)


def to_cpu(array):
    if GPU_AVAILABLE:
        return cp.asnumpy(array)
    return np.asarray(array)


def save_and_display_animation(
    animation,
    stem,
    fps=60,
    dpi=150,
    bitrate=10000,
):
    """Guarda y muestra automáticamente una animación."""
    if shutil.which("ffmpeg"):
        output = ANIM_DIR / f"{stem}.mp4"
        writer = FFMpegWriter(
            fps=fps,
            bitrate=bitrate,
            metadata={"title": stem},
        )
        animation.save(output, writer=writer, dpi=dpi)
        display(Video(str(output), embed=True))
    else:
        output = ANIM_DIR / f"{stem}.gif"
        writer = PillowWriter(fps=min(fps, 35))
        animation.save(
            output,
            writer=writer,
            dpi=min(dpi, 110),
        )
        display(Image(filename=str(output)))

    print("Animación guardada en:", output.resolve())
    return output

## Simulación 3.7.A — Una envolvente de Perron en el disco

Sea

$$
g(\theta)
=
e^{0.55\cos\theta}
+
0.18\cos(4\theta)
-
0.12\sin(7\theta).
$$

Para cada truncamiento de Fourier $s_N$, definimos

$$
\varepsilon_N
=
\max_\theta\left(s_N(\theta)-g(\theta)\right)+10^{-8}
$$

y tomamos la extensión armónica de

$$
s_N-\varepsilon_N.
$$

La función resultante $v_N$ es armónica y satisface

$$
v_N\leq g
\qquad
\text{sobre }\partial B_1.
$$

Por tanto, cada $v_N$ es una subfunción de Perron. La envolvente finita

$$
H_N(x)
=
\max\{v_0(x),\dots,v_N(x)\}
$$

es subarmónica y crece hacia la solución de Poisson.

In [ ]:
# ============================================================
# FAMILIA NUMÉRICA DE SUBFUNCIONES DE PERRON
# ============================================================

n_theta = 8192 if GPU_AVAILABLE else 4096
theta_boundary = xp.linspace(
    -math.pi,
    math.pi,
    n_theta,
    endpoint=False,
)

g_boundary = (
    xp.exp(0.55 * xp.cos(theta_boundary))
    + 0.18 * xp.cos(4.0 * theta_boundary)
    - 0.12 * xp.sin(7.0 * theta_boundary)
)

n_reference = 120
mode_numbers = xp.arange(
    1,
    n_reference + 1,
    dtype=xp.float64,
)

a0 = xp.mean(g_boundary)
a_coeff = 2.0 * xp.mean(
    g_boundary[None, :]
    * xp.cos(
        mode_numbers[:, None]
        * theta_boundary[None, :]
    ),
    axis=1,
)
b_coeff = 2.0 * xp.mean(
    g_boundary[None, :]
    * xp.sin(
        mode_numbers[:, None]
        * theta_boundary[None, :]
    ),
    axis=1,
)

grid_n = 500 if GPU_AVAILABLE else 300
axis = xp.linspace(-1.0, 1.0, grid_n)
X, Y = xp.meshgrid(axis, axis, indexing="xy")
R = xp.sqrt(X**2 + Y**2)
Theta = xp.arctan2(Y, X)
disk_mask = R <= 1.0

# Construcción incremental de la extensión exacta truncada.
exact_interior = xp.full_like(R, a0)
radial_power = xp.ones_like(R)

for mode in range(1, n_reference + 1):
    radial_power = radial_power * R
    exact_interior = exact_interior + radial_power * (
        a_coeff[mode - 1] * xp.cos(mode * Theta)
        + b_coeff[mode - 1] * xp.sin(mode * Theta)
    )

exact_interior = xp.where(
    disk_mask,
    exact_interior,
    xp.nan,
)

# Truncamientos usados como subfunciones.
candidate_orders = np.unique(
    np.concatenate(
        [
            np.arange(0, 21),
            np.array([24, 28, 32, 38, 46, 56, 70, 90, 120]),
        ]
    )
)

partial_boundary = xp.full_like(
    theta_boundary,
    a0,
)
partial_interior = xp.full_like(R, a0)
radial_power = xp.ones_like(R)

candidate_frames = []
envelope_frames = []
boundary_violations = []
shifts = []

envelope = xp.full_like(R, -xp.inf)

order_set = set(int(value) for value in candidate_orders)
maximum_order = int(candidate_orders[-1])

for mode in range(0, maximum_order + 1):
    if mode > 0:
        radial_power = radial_power * R

        partial_boundary = partial_boundary + (
            a_coeff[mode - 1]
            * xp.cos(mode * theta_boundary)
            + b_coeff[mode - 1]
            * xp.sin(mode * theta_boundary)
        )

        partial_interior = partial_interior + radial_power * (
            a_coeff[mode - 1] * xp.cos(mode * Theta)
            + b_coeff[mode - 1] * xp.sin(mode * Theta)
        )

    if mode in order_set:
        shift = (
            xp.max(partial_boundary - g_boundary)
            + 1e-8
        )
        candidate = partial_interior - shift
        candidate = xp.where(
            disk_mask,
            candidate,
            xp.nan,
        )

        envelope = xp.where(
            disk_mask,
            xp.maximum(envelope, candidate),
            xp.nan,
        )

        boundary_candidate = partial_boundary - shift
        violation = xp.max(
            boundary_candidate - g_boundary
        )

        candidate_frames.append(
            to_cpu(candidate).astype(np.float32)
        )
        envelope_frames.append(
            to_cpu(envelope).astype(np.float32)
        )
        boundary_violations.append(
            float(to_cpu(violation))
        )
        shifts.append(float(to_cpu(shift)))

candidate_frames = np.asarray(candidate_frames)
envelope_frames = np.asarray(envelope_frames)
exact_cpu = to_cpu(exact_interior)

print("Número de candidatos:", len(candidate_orders))
print(
    "Máxima violación de la condición de frontera:",
    max(boundary_violations),
)
print(
    "Error final de la envolvente en el centro:",
    float(
        exact_cpu[grid_n // 2, grid_n // 2]
        - envelope_frames[-1, grid_n // 2, grid_n // 2]
    ),
)

In [ ]:
# Animación de la envolvente creciente.

finite_exact = exact_cpu[np.isfinite(exact_cpu)]
vmin = float(np.min(finite_exact))
vmax = float(np.max(finite_exact))

fig, ax = plt.subplots(figsize=(8.5, 7.1))

image = ax.imshow(
    envelope_frames[0],
    origin="lower",
    extent=[-1.0, 1.0, -1.0, 1.0],
    interpolation="bilinear",
    vmin=vmin,
    vmax=vmax,
)
fig.colorbar(image, ax=ax, label=r"$H_N(x)$")

circle = plt.Circle(
    (0.0, 0.0),
    1.0,
    fill=False,
    linewidth=1.5,
)
ax.add_patch(circle)

ax.set_aspect("equal")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title("Envolvente creciente de subfunciones")

status = ax.text(
    0.02,
    0.97,
    "",
    transform=ax.transAxes,
    va="top",
    bbox={"boxstyle": "round", "alpha": 0.8},
)


def update_envelope(frame):
    image.set_data(envelope_frames[frame])
    status.set_text(
        rf"$N={candidate_orders[frame]}$"
        + "\n"
        + rf"$\varepsilon_N={shifts[frame]:.3e}$"
    )
    return image, status


animation = FuncAnimation(
    fig,
    update_envelope,
    frames=len(envelope_frames),
    interval=1000.0 / 30.0,
    blit=False,
)

fig.tight_layout()

save_and_display_animation(
    animation,
    "03.7.A_envolvente_perron",
    fps=30,
    dpi=170,
    bitrate=14000,
)

plt.close(fig)

In [ ]:
# Comparación final entre la envolvente y la solución de Poisson.

error = exact_cpu - envelope_frames[-1]
error = np.where(
    np.isfinite(exact_cpu),
    error,
    np.nan,
)

fig, ax = plt.subplots(figsize=(8.5, 7.0))
image = ax.imshow(
    error,
    origin="lower",
    extent=[-1.0, 1.0, -1.0, 1.0],
    interpolation="bilinear",
)
fig.colorbar(
    image,
    ax=ax,
    label=r"$H_g-H_N$",
)
circle = plt.Circle(
    (0.0, 0.0),
    1.0,
    fill=False,
    linewidth=1.5,
)
ax.add_patch(circle)
ax.set_aspect("equal")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title("Error de la envolvente finita")
fig.tight_layout()

path = FIG_DIR / "03.7.A_error_envolvente.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print("Error máximo:", float(np.nanmax(error)))
print("Error medio:", float(np.nanmean(error)))
print(path.resolve())

## Simulación 3.7.B — Barrera y recuperación en un punto regular

En el disco unitario consideramos

$$
\xi=(1,0).
$$

La bola exterior de centro $z=(2,0)$ y radio $1$ toca al disco únicamente en
$\xi$. La función

$$
w(x)=\log|x-z|
$$

es armónica en el disco, positiva en su interior y satisface

$$
w(x)\to0
\qquad
\text{cuando }x\to\xi.
$$

Se compara el error de la solución de Poisson sobre el radio que termina en
$\xi$ con la desaparición de la barrera.

In [ ]:
# ============================================================
# BARRERA EN xi=(1,0)
# ============================================================

barrier = xp.log(
    xp.sqrt(
        (X - 2.0) ** 2
        + Y**2
    )
)
barrier = xp.where(
    disk_mask,
    barrier,
    xp.nan,
)
barrier_cpu = to_cpu(barrier)

fig, ax = plt.subplots(figsize=(8.5, 7.0))
image = ax.imshow(
    barrier_cpu,
    origin="lower",
    extent=[-1.0, 1.0, -1.0, 1.0],
    interpolation="bilinear",
)
fig.colorbar(image, ax=ax, label=r"$w(x)$")

domain_circle = plt.Circle(
    (0.0, 0.0),
    1.0,
    fill=False,
    linewidth=1.5,
)
exterior_circle = plt.Circle(
    (2.0, 0.0),
    1.0,
    fill=False,
    linestyle="--",
    linewidth=1.5,
)
ax.add_patch(domain_circle)
ax.add_patch(exterior_circle)
ax.plot([1.0], [0.0], marker="o", label=r"$\xi$")
ax.set_aspect("equal")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title("Barrera producida por una esfera exterior")
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.7.B_barrera_esfera_exterior.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)
print(path.resolve())

In [ ]:
# Recuperación radial del dato en xi=(1,0).

r_line = np.linspace(0.0, 0.9995, 700)
theta_xi = 0.0

a0_cpu = float(to_cpu(a0))
a_cpu = to_cpu(a_coeff)
b_cpu = to_cpu(b_coeff)

u_line = np.full_like(r_line, a0_cpu)

for mode in range(1, n_reference + 1):
    u_line += r_line**mode * (
        a_cpu[mode - 1] * math.cos(mode * theta_xi)
        + b_cpu[mode - 1] * math.sin(mode * theta_xi)
    )

g_xi = (
    math.exp(0.55)
    + 0.18
)
boundary_error = np.abs(u_line - g_xi)
barrier_line = np.log(2.0 - r_line)

fig, ax = plt.subplots(figsize=(9, 5.7))
ax.semilogy(
    1.0 - r_line,
    boundary_error,
    label=r"$|u(r\xi)-g(\xi)|$",
)
ax.semilogy(
    1.0 - r_line,
    barrier_line,
    label=r"$w(r\xi)$",
)
ax.set_xlabel(r"$1-r$")
ax.set_ylabel("magnitud")
ax.set_title("Recuperación del dato en un punto regular")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.7.B_recuperacion_punto_regular.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

# 3.7.1 Subfunciones, superfunciones y familias de Perron

## **Complemento para cerrar el temario oficial**

Sea $\Omega\subset\mathbb R^n$ un dominio acotado y sea

$$
g\in C(\partial\Omega).
$$

## **Definición 3.7.1 (Subfunción de Perron).**

Una función $v$ es una **subfunción para el dato $g$** si:

1. $v$ es subarmónica en $\Omega$;
2. $v$ está acotada superiormente en $\Omega$;
3. para todo $\xi\in\partial\Omega$,

   $$
   \limsup_{\substack{x\to\xi\\x\in\Omega}}v(x)
   \leq
   g(\xi).
   $$

La familia de todas las subfunciones se denota por

$$
\mathcal S_g.
$$

## **Definición 3.7.2 (Superfunción de Perron).**

Una función $V$ es una **superfunción para el dato $g$** si:

1. $V$ es superarmónica en $\Omega$;
2. $V$ está acotada inferiormente;
3. para todo $\xi\in\partial\Omega$,

   $$
   \liminf_{\substack{x\to\xi\\x\in\Omega}}V(x)
   \geq
   g(\xi).
   $$

La familia correspondiente se denota por

$$
\mathcal U_g.
$$

## **Definición 3.7.3 (Funciones superior e inferior de Perron).**

La **función superior de Perron** es

$$
\overline H_g(x)
=
\sup_{v\in\mathcal S_g}v(x).
$$

La **función inferior de Perron** es

$$
\underline H_g(x)
=
\inf_{V\in\mathcal U_g}V(x).
$$

## **Proposición 3.7.4 (No vacuidad y cotas elementales).**

Las familias $\mathcal S_g$ y $\mathcal U_g$ no son vacías. Además,

$$
\min_{\partial\Omega}g
\leq
\overline H_g
\leq
\underline H_g
\leq
\max_{\partial\Omega}g.
$$

### Demostración

Las constantes

$$
m=\min_{\partial\Omega}g
$$

y

$$
M=\max_{\partial\Omega}g
$$

son, respectivamente, una subfunción y una superfunción.

Sea $v\in\mathcal S_g$ y $V\in\mathcal U_g$. La diferencia $v-V$ es
subarmónica en el sentido de comparación. Sus límites superiores en la frontera
son no positivos. El principio de comparación implica

$$
v\leq V
$$

en $\Omega$. Tomando supremo en $v$ e ínfimo en $V$ se obtiene

$$
\overline H_g\leq\underline H_g.
$$

Las cotas por $m$ y $M$ se siguen de las funciones constantes.

$\square$

### **Observación 3.7.5.**

El método no comienza suponiendo que existe una solución armónica con dato $g$.
Construye primero una envolvente ordenada de funciones subarmónicas permitidas.

### Ejercicios — Sección 3.7.1

> **Ruta corta de estudio:** dos ejercicios de técnica y una variación de nivel Examen General.

1. Verifique que toda solución clásica del problema de Dirichlet pertenece
   simultáneamente a $\mathcal S_g$ y a $\mathcal U_g$.

2. Demuestre que si $g_1\leq g_2$, entonces

   $$
   \overline H_{g_1}\leq\overline H_{g_2}.
   $$

3. **Tipo Examen General.** Explique por qué el método de Perron transforma un
   problema de existencia en un problema de demostrar regularidad y armonicidad
   de una envolvente.


# 3.7.2 Máximos y elevación armónica

## **Lema 3.7.6 (Máximo de subfunciones).**

Si $v_1,v_2\in\mathcal S_g$, entonces

$$
v=\max\{v_1,v_2\}
$$

también pertenece a $\mathcal S_g$.

### Demostración

El máximo de dos funciones subarmónicas es subarmónico. Además, $v$ sigue
acotada superiormente y

$$
\limsup_{x\to\xi}v(x)
\leq
\max\left\{
\limsup_{x\to\xi}v_1(x),
\limsup_{x\to\xi}v_2(x)
\right\}
\leq g(\xi).
$$

$\square$

## **Definición 3.7.7 (Elevación armónica en una bola).**

Sea

$$
\overline B\subset\Omega
$$

y sea $v\in\mathcal S_g$. Sea $h$ la solución del problema de Dirichlet

$$
\begin{cases}
\Delta h=0,
& \text{en }B,\\
h=v,
& \text{sobre }\partial B.
\end{cases}
$$

La **elevación armónica** de $v$ en $B$ es

$$
v^B(x)
=
\begin{cases}
h(x),
& x\in B,\\
v(x),
& x\in\Omega\setminus B.
\end{cases}
$$

## **Lema 3.7.8 (Propiedades de la elevación armónica).**

Bajo las hipótesis anteriores:

1. $v^B\in\mathcal S_g$;
2. $v^B\geq v$ en $\Omega$;
3. $v^B$ es armónica en $B$.

### Demostración añadida

Como $v$ es subarmónica en $B$ y $h=v$ sobre $\partial B$, el principio de
comparación da

$$
v\leq h
\qquad
\text{en }B.
$$

Por tanto, $v^B\geq v$. El lema de pegado para funciones subarmónicas muestra
que $v^B$ es subarmónica en todo $\Omega$. Fuera de $B$ no cambia, por lo que
conserva la condición de frontera y la cota superior.

$\square$

### **Interpretación.**

La elevación reemplaza localmente una subfunción por la mayor función armónica
con los mismos valores sobre la frontera de la bola. Es el mecanismo que obliga
a la envolvente final a ser armónica.

### Ejercicios — Sección 3.7.2

> **Ruta corta de estudio:** dos ejercicios de técnica y una variación de nivel Examen General.

1. Demuestre el lema de pegado utilizado en la prueba.

2. Verifique que la elevación armónica no modifica la condición límite sobre
   $\partial\Omega$.

3. **Tipo Examen General.** Explique por qué tomar únicamente el supremo de
   funciones subarmónicas no garantiza por sí mismo que el resultado sea
   armónico y qué papel desempeña la elevación.


# 3.7.3 Armonicidad de la envolvente

## **Teorema 3.7.9 (Teorema de Perron).**

La función superior de Perron

$$
\overline H_g(x)
=
\sup_{v\in\mathcal S_g}v(x)
$$

es armónica en $\Omega$.

### Demostración añadida

Fijemos $x_0\in\Omega$ y una bola

$$
\overline B\subset\Omega
$$

que contiene a $x_0$.

Por la definición de supremo, existe una sucesión
$\{v_j\}\subset\mathcal S_g$ tal que

$$
v_j(x_0)\to\overline H_g(x_0).
$$

Usando repetidamente el Lema 3.7.6, podemos suponer que $v_j$ es creciente.
Sustituimos cada $v_j$ por su elevación armónica en $B$ y la denotamos por
$h_j$. Entonces:

- $h_j$ es armónica en $B$;
- $h_j\in\mathcal S_g$;
- $h_j$ es creciente;
- $h_j\leq\max_{\partial\Omega}g$.

Por el teorema de convergencia de Harnack, la sucesión converge localmente de
manera uniforme en $B$ a una función armónica $h$. Además,

$$
h\leq\overline H_g
$$

y

$$
h(x_0)=\overline H_g(x_0).
$$

Falta probar que $h=\overline H_g$ en toda la bola. Sea
$v\in\mathcal S_g$. Definimos

$$
w_j=\max\{v,h_j\}
$$

y elevamos $w_j$ armónicamente en $B$, obteniendo $k_j$. Entonces

$$
k_j\geq h_j
$$

y

$$
k_j\leq\overline H_g.
$$

En $x_0$,

$$
0
\leq
k_j(x_0)-h_j(x_0)
\leq
\overline H_g(x_0)-h_j(x_0)
\to0.
$$

La diferencia $k_j-h_j$ es armónica y no negativa en $B$. La desigualdad de
Harnack implica que converge a cero uniformemente en compactos. Como

$$
k_j\geq w_j\geq v,
$$

se obtiene

$$
h\geq v
$$

en $B$. Tomando supremo en todas las subfunciones,

$$
h\geq\overline H_g.
$$

Por tanto,

$$
h=\overline H_g
$$

en $B$. Como $x_0$ es arbitrario, $\overline H_g$ es armónica en $\Omega$.

$\square$

## **Corolario 3.7.10 (Armonicidad de la función inferior).**

La función inferior de Perron $\underline H_g$ es armónica en $\Omega$.

### Demostración

Se usa la identidad

$$
\underline H_g
=
-\overline H_{-g}.
$$

$\square$

## **Definición 3.7.11 (Dato resolutivo).**

El dato $g$ se llama **resolutivo** si

$$
\overline H_g=\underline H_g.
$$

En ese caso, la función común se denota por $H_g$ y se llama
**solución de Perron**.

### Ejercicios — Sección 3.7.3

> **Ruta corta de estudio:** dos ejercicios de técnica y una variación de nivel Examen General.

1. Enuncie con precisión el teorema de convergencia de Harnack utilizado en la
   demostración.

2. Justifique por qué la sucesión de elevaciones puede elegirse creciente.

3. **Tipo Examen General.** Describa la demostración de que la envolvente de
   Perron es armónica, indicando los tres ingredientes esenciales.


# 3.7.4 Barreras y puntos regulares

## **Definición 3.7.12 (Barrera en un punto).**

Sea $\xi\in\partial\Omega$. Una función $w$ es una barrera en $\xi$ si existe
un entorno $V$ de $\xi$ tal que:

1. $w$ es superarmónica en $\Omega\cap V$;
2. $w>0$ en $\Omega\cap V$;
3. $w(x)\to0$ cuando $x\to\xi$ desde $\Omega$;
4. $w$ permanece separada de cero en la parte de la frontera de
   $\Omega\cap V$ que está alejada de $\xi$.

## **Definición 3.7.13 (Punto regular).**

Un punto $\xi\in\partial\Omega$ es regular si, para todo
$g\in C(\partial\Omega)$,

$$
\lim_{\substack{x\to\xi\\x\in\Omega}}
H_g(x)
=
g(\xi).
$$

## **Teorema 3.7.14 (Criterio de barrera).**

Si existe una barrera en $\xi\in\partial\Omega$, entonces $\xi$ es regular.

### Demostración añadida

Sea $\varepsilon>0$. Por continuidad de $g$, existe un entorno $W$ de $\xi$
tal que

$$
|g(\eta)-g(\xi)|<\varepsilon
$$

para $\eta\in\partial\Omega\cap W$.

Sea $w$ una barrera en $\xi$. Como $w$ está separada de cero fuera de $W$,
podemos elegir $C>0$ suficientemente grande para que

$$
g(\xi)-\varepsilon-Cw
$$

sea una subfunción y

$$
g(\xi)+\varepsilon+Cw
$$

sea una superfunción.

Por la definición de las funciones de Perron,

$$
g(\xi)-\varepsilon-Cw(x)
\leq
\overline H_g(x)
\leq
\underline H_g(x)
\leq
g(\xi)+\varepsilon+Cw(x).
$$

Cuando $x\to\xi$,

$$
w(x)\to0.
$$

Por tanto,

$$
g(\xi)-\varepsilon
\leq
\liminf_{x\to\xi}\overline H_g(x)
\leq
\limsup_{x\to\xi}\underline H_g(x)
\leq
g(\xi)+\varepsilon.
$$

Haciendo $\varepsilon\downarrow0$ se concluye

$$
\overline H_g(x),
\underline H_g(x)
\longrightarrow
g(\xi).
$$

En particular, ambas funciones coinciden y recuperan el dato en $\xi$.

$\square$

## **Corolario 3.7.15 (Condición de esfera exterior).**

Si $\Omega$ satisface la condición de esfera exterior en $\xi$, entonces $\xi$
es regular.

### Complemento

Las barreras explícitas son:

si $n=2$,

$$
w(x)
=
\log\frac{|x-z|}{R};
$$

si $n\geq3$,

$$
w(x)
=
R^{2-n}-|x-z|^{2-n},
$$

donde $\overline{B_R(z)}$ es la esfera exterior tangente en $\xi$.

### Ejercicios — Sección 3.7.4

> **Ruta corta de estudio:** dos ejercicios de técnica y una variación de nivel Examen General.

1. Verifique cuidadosamente las desigualdades de frontera usadas en la prueba
   del criterio de barrera.

2. Construya una barrera para cada punto de la frontera de una bola.

3. **Tipo Examen General.** Explique la relación entre barrera, punto regular y
   recuperación del dato continuo. Incluya una fórmula explícita de barrera.


# 3.7.5 Existencia clásica mediante Perron

## **Teorema 3.7.16 (Solvencia de Perron en dominios regulares).**

Sea $\Omega\subset\mathbb R^n$ un dominio acotado. Supongamos que todo punto de
$\partial\Omega$ es regular. Entonces, para cada

$$
g\in C(\partial\Omega),
$$

existe una única función

$$
u\in C^2(\Omega)\cap C(\overline\Omega)
$$

tal que

$$
\begin{cases}
\Delta u=0,
& \text{en }\Omega,\\
u=g,
& \text{sobre }\partial\Omega.
\end{cases}
$$

La solución es la función de Perron:

$$
u=H_g.
$$

### Demostración

El Teorema 3.7.9 muestra que $\overline H_g$ es armónica. El criterio de
regularidad implica que, en cada punto de frontera,

$$
\overline H_g(x)\to g(\xi).
$$

Así, $\overline H_g$ se extiende continuamente a $\overline\Omega$ con dato
$g$. La unicidad se sigue del principio del máximo.

$\square$

## **Corolario 3.7.17.**

Si $\Omega$ es acotado y satisface la condición de esfera exterior en cada punto
de la frontera, entonces el problema de Dirichlet tiene una única solución
clásica para cada dato continuo.

## **Corolario 3.7.18.**

Todo dominio acotado con frontera de clase $C^2$ es regular para el problema de
Dirichlet.

# 3.7.6 Respuesta conceptual para el Examen General

## **Proposición 3.7.19 (Esquema del método de Perron).**

El método de Perron sirve para demostrar existencia de soluciones armónicas del
problema de Dirichlet sin construir primero una función de Green.

El procedimiento es:

1. reunir todas las funciones subarmónicas que permanecen por debajo del dato
   en la frontera;
2. tomar su envolvente superior;
3. usar máximos y elevaciones armónicas para demostrar que la envolvente es
   armónica;
4. usar barreras para demostrar que la envolvente recupera el dato en los puntos
   regulares;
5. concluir existencia clásica cuando todos los puntos de frontera son
   regulares;
6. usar el principio del máximo para obtener unicidad.

### Diferencia entre los dos problemas centrales

- La **armonicidad interior** de la envolvente se obtiene mediante elevaciones y
  Harnack.
- La **continuidad hasta la frontera** depende de la regularidad geométrica de
  los puntos de frontera y se controla mediante barreras.

### Ejercicios integradores — Método de Perron

> **Ruta corta de estudio:** dos ejercicios de técnica y una variación de nivel Examen General.

1. Dé una respuesta de una página a la pregunta:

   > ¿Para qué sirve el método de Perron y en qué consiste?

   Incluya las palabras subfunción, envolvente, elevación armónica, barrera y
   punto regular.

2. Demuestre que la solución de Perron depende monótonamente del dato de
   frontera.

3. **Tipo Examen General.** Sea $\Omega$ un dominio acotado con condición de
   esfera exterior. Demuestre existencia y unicidad para el problema de
   Dirichlet con dato continuo usando el método de Perron.


# Control de cobertura y estado del capítulo

## Contenido cubierto

- subfunciones y superfunciones;
- funciones superior e inferior de Perron;
- no vacuidad y comparación;
- estabilidad bajo máximos;
- elevación armónica;
- armonicidad de la envolvente;
- barreras y puntos regulares;
- recuperación del dato;
- existencia y unicidad en dominios regulares;
- condición de esfera exterior;
- respuesta conceptual de nivel Examen General;
- visualización numérica de envolventes.

## Relación con las fuentes

La convención de subarmonicidad proviene de las notas. El desarrollo completo del
método de Perron fue añadido porque constituye un apartado explícito del temario
oficial y aparece como pregunta conceptual en un examen histórico.

## Pendiente inmediato

El siguiente notebook de la cola es

$$
\texttt{03.8.01\_Energia\_y\_principio\_de\_Dirichlet.ipynb}.
$$

Se desarrollarán la identidad de energía, formulación variacional, minimización
del funcional de Dirichlet, ecuación de Euler-Lagrange y unicidad.